<a href="https://colab.research.google.com/github/mvishrutha/ADA-LAB-M-Vishrutha/blob/main/Week-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import numpy as np

# ==========================================
# 1. PARAMETERS & PROBLEM SETUP
# ==========================================
NUM_STOPS = 5          # Number of delivery locations (excluding the depot)
POP_SIZE = 30          # Number of route configurations in the population
GENERATIONS = 100      # How many generations to evolve through
MUTATION_RATE = 0.15   # Probability of a route mutating

VEHICLE_BASE_WEIGHT = 1500  # kg (Standard car/SUV weight)

# Cargo weight (in kg) to be dropped off at each specific stop
CARGO_WEIGHTS = {
    1: 200,
    2: 450,  # Very heavy cargo
    3: 150,
    4: 350,  # Heavy cargo
    5: 100
}

# Environmental Data Matrix between stops (Depot is index 0)
# Format: (distance_in_km, average_speed_kmh, road_gradient_degrees)
# Notice that some roads are hilly (positive gradient) or slow city zones (30-40 km/h)
ENV_MAP = {
    (0, 1): (10, 40, 3),   (0, 2): (25, 80, 0),   (0, 3): (15, 50, 1),   (0, 4): (30, 85, -1),  (0, 5): (12, 30, 0),
    (1, 0): (10, 40, -3),  (1, 2): (12, 60, 4),   (1, 3): (8, 40, 0),    (1, 4): (20, 70, 2),   (1, 5): (18, 50, -1),
    (2, 0): (25, 80, 0),   (2, 1): (12, 60, -4),  (2, 3): (10, 55, -2),  (2, 4): (15, 80, 0),   (2, 5): (22, 65, 1),
    (3, 0): (15, 50, -1),  (3, 1): (8, 40, 0),    (3, 2): (10, 55, 2),   (3, 4): (18, 60, 3),   (3, 5): (14, 45, 0),
    (4, 0): (30, 85, 1),   (4, 1): (20, 70, -2),  (4, 2): (15, 80, 0),   (4, 3): (18, 60, -3),  (4, 5): (25, 75, 1),
    (5, 0): (12, 30, 0),   (5, 1): (18, 50, 1),   (5, 2): (22, 65, -1),  (5, 3): (14, 45, 0),   (5, 4): (25, 75, -1)
}

def get_road_data(u, v):
    """Retrieves distance, speed, and slope between two locations."""
    if (u, v) in ENV_MAP:
        return ENV_MAP[(u, v)]
    elif (v, u) in ENV_MAP:
        dist, spd, grad = ENV_MAP[(v, u)]
        return (dist, spd, -grad)  # Reverse the hill incline if driving backwards
    return (15, 50, 0)  # Default fallback if data missing

# ==========================================
# 2. PHYSICAL FUEL CONSUMPTION ENGINE
# ==========================================
def calculate_fuel_burn(u, v, current_weight):
    """
    Calculates estimated fuel consumed (in Litres) for a single leg of the trip
    using core engineering principles (rolling resistance, gravity, and drag).
    """
    dist_km, speed_kmh, gradient_deg = get_road_data(u, v)

    # Convert speed to meters per second
    v_mps = speed_kmh / 3.6

    # Physics Forces: Rolling Resistance + Gravity pulling vehicle back on hills
    gravity_acceleration = 9.81
    rolling_resistance_coef = 0.015

    angle_rad = np.radians(gradient_deg)

    force_gravity = current_weight * gravity_acceleration * np.sin(angle_rad)
    force_rolling = current_weight * gravity_acceleration * np.cos(angle_rad) * rolling_resistance_coef

    # Total chassis force required (Cannot be negative; brakes absorb downhills)
    propulsion_force = max(0.0, force_rolling + force_gravity)

    # Aerodynamic Drag Force: 0.5 * Drag Coef * Frontal Area * Air Density * V^2
    force_drag = 0.5 * 0.32 * 2.3 * 1.2 * (v_mps ** 2)

    # Power required at the wheels (Watts)
    power_watts = (propulsion_force + force_drag) * v_mps

    # Calculate travel time in hours
    travel_time_hours = dist_km / speed_kmh if speed_kmh > 0 else 0

    # Empirical fuel calculation factor mapping mechanical work to chemical fuel burn
    # Ensures a baseline idle/cruise fuel burn is preserved even on steep downhills
    fuel_litres = max(0.06 * dist_km, (power_watts * 0.000085) * travel_time_hours)

    return fuel_litres

def evaluate_fitness(route):
    """
    Simulates driving the route. Tracks structural vehicle weight drops
    after every drop-off to compute absolute fuel metrics.
    """
    total_fuel = 0.0
    total_distance = 0.0

    # Total initial vehicle mass = empty car + all cargo
    current_weight = VEHICLE_BASE_WEIGHT + sum(CARGO_WEIGHTS.values())

    # Step 1: Start at Depot (0) -> Head to first destination
    current_location = 0

    for next_stop in route:
        total_fuel += calculate_fuel_burn(current_location, next_stop, current_weight)
        total_distance += get_road_data(current_location, next_stop)[0]

        # Drop cargo payload off -> Car becomes lighter for next leg
        current_weight -= CARGO_WEIGHTS[next_stop]
        current_location = next_stop

    # Step 2: Last destination -> Complete the loop back to Depot (0)
    total_fuel += calculate_fuel_burn(current_location, 0, current_weight)
    total_distance += get_road_data(current_location, 0)[0]

    return total_fuel, total_distance

# ==========================================
# 3. GENETIC ALGORITHM OPERATORS
# ==========================================
def generate_initial_population():
    """Generates random valid route permutations."""
    population = []
    base_nodes = list(range(1, NUM_STOPS + 1))
    for _ in range(POP_SIZE):
        random.shuffle(base_nodes)
        population.append(list(base_nodes))
    return population

def ordered_crossover(parent1, parent2):
    """Breeds two parent routes to create a child without creating duplicate stops."""
    size = len(parent1)
    child = [None] * size

    # Select random segment from parent 1
    start, end = sorted(random.sample(range(size), 2))
    child[start:end] = parent1[start:end]

    # Fill remaining slots using parent 2 sequence order
    fill_pointer = 0
    for stop in parent2:
        if stop not in child:
            while child[fill_pointer] is not None:
                fill_pointer += 1
            child[fill_pointer] = stop

    return child

def apply_mutation(route):
    """Randomly swaps two stops to maintain genetic diversity."""
    if random.random() < MUTATION_RATE:
        idx1, idx2 = random.sample(range(len(route)), 2)
        route[idx1], route[idx2] = route[idx2], route[idx1]
    return route

# ==========================================
# 4. EVOLUTION EXECUTION ENGINE
# ==========================================
def optimize_vehicle_route():
    population = generate_initial_population()

    global_best_route = None
    global_minimum_fuel = float('inf')
    global_associated_distance = 0.0

    print("Beginning Evolution Process...")

    for generation in range(1, GENERATIONS + 1):
        scored_generation = []

        # Evaluate every route profile in the current generation
        for route in population:
            fuel_cost, distance_cost = evaluate_fitness(route)
            scored_generation.append((fuel_cost, distance_cost, route))

            # Keep track of the absolute best route ever discovered
            if fuel_cost < global_minimum_fuel:
                global_minimum_fuel = fuel_cost
                global_best_route = route
                global_associated_distance = distance_cost

        # Sort current routes so the most fuel-efficient appear first
        scored_generation.sort(key=lambda x: x[0])

        # Selection: Retain the best half of the population (Elitism)
        survivors = [item[2] for item in scored_generation[:POP_SIZE // 2]]

        # Replenish population through Crossover and Mutation
        next_gen_pool = list(survivors)
        while len(next_gen_pool) < POP_SIZE:
            parent_a, parent_b = random.sample(survivors, 2)
            child_route = ordered_crossover(parent_a, parent_b)
            child_route = apply_mutation(child_route)
            next_gen_pool.append(child_route)

        population = next_gen_pool

        # Status update every 20 generations
        if generation % 20 == 0 or generation == 1:
            print(f" > Generation {generation:03d}: Current Lowest Fuel = {scored_generation[0][0]:.2f} L")

    # ==========================================
    # 5. DISPLAY FINAL OPTIMIZED RESULT
    # ==========================================
    print("\n=============================================")
    print("        OPTIMIZATION SUCCESSFUL!!            ")
    print("=============================================")
    print(f"Optimal Driving Route  : Depot -> {' -> '.join(map(str, global_best_route))} -> Depot")
    print(f"Minimised Fuel Economy : {global_minimum_fuel:.2f} Litres")
    print(f"Total Odometer Driven  : {global_associated_distance:.2f} km")
    print("=============================================\n")

if __name__ == "__main__":
    optimize_vehicle_route()


Beginning Evolution Process...
 > Generation 001: Current Lowest Fuel = 4.86 L
 > Generation 020: Current Lowest Fuel = 4.80 L
 > Generation 040: Current Lowest Fuel = 4.80 L
 > Generation 060: Current Lowest Fuel = 4.80 L
 > Generation 080: Current Lowest Fuel = 4.80 L
 > Generation 100: Current Lowest Fuel = 4.80 L

        OPTIMIZATION SUCCESSFUL!!            
Optimal Driving Route  : Depot -> 5 -> 4 -> 2 -> 3 -> 1 -> Depot
Minimised Fuel Economy : 4.80 Litres
Total Odometer Driven  : 80.00 km

